# Phase 4 - AttentiveFP, the external baseline

the study plan section 7 asks for two standard external baselines. This runs the
first. The second, Chemprop, cannot share an environment with this one -- it pins its own
torch -- so it runs separately via `src/baselines/chemprop_runner.py`.

**What makes this a fair comparison.** AttentiveFP is wrapped as an *encoder*
(`src/models/encoders/graph.py`), so it trains through the same loop, the same splits, the
same 256-d projection and the same head as every other view here. A baseline trained under
its own author's recipe and compared against ours would confound the architecture with the
training protocol, which is the exact failure this project exists to document. What stays
genuinely AttentiveFP is the GRU-based message passing over attended neighbourhoods and the
`num_timesteps` rounds of graph-level attention pooling.

**Bundle: `mpp_fusion_bundle.zip`** (18 MB, already rebuilt with current code). It carries
graphs and fingerprints, which is all a graph view needs.

**Read this before comparing results.** Session 20 established that the same code, seed and
splits on a different device produce a different model -- 18% of *single-split* numbers move
by more than the minimum detectable effect, though the five-split mean absorbs it. This run
is on a T4, and the committed `gin_ref`, `gine` and `desc` rows are CPU. Compare against
`gin_ref_gpu` (also T4) for a within-device comparison.

Expected: **~3-4 h on a T4**. `--resume` continues after a timeout.

## 1. Check you actually got a GPU

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.')

## 2. Connect your Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_fusion_bundle.zip'  # edit if elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_phase4'

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} -- check path, re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')

In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_phase3_regen.zip'  # edit if elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_phase3'

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} -- check path, re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')

## 4. Unpack and install

`torchao` is uninstalled deliberately: an old build's probe *raises* instead of returning
False, which kills every run that touches `peft`.

In [ ]:
import zipfile, os

WORK = '/content/mpp'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)

!pip -q install peft accelerate torch_geometric
!pip -q uninstall -y torchao

import torch_geometric
print('torch_geometric', torch_geometric.__version__)

## 5. Keep results on Drive

`results/runs` becomes a symlink into Drive, so a disconnect loses nothing and `--resume`
can see what already finished. Note this is a **different folder** from the end-to-end
run's, so the two cannot collide.

In [ ]:
import os

os.makedirs(f'{OUTDIR}/runs', exist_ok=True)
os.makedirs('results', exist_ok=True)
if not os.path.islink('results/runs'):
    if os.path.exists('results/runs'):
        import shutil; shutil.rmtree('results/runs')
    os.symlink(f'{OUTDIR}/runs', 'results/runs')
print('results/runs ->', os.path.realpath('results/runs'))

## 6. Smoke test - do not skip this

Three epochs on the smallest dataset. AttentiveFP needs `edge_attr` at the right width even
for a molecule with no bonds, and that guard is worth exercising before three hours of
training rather than after.

In [ ]:
!python -m src.data.materialize --variant deepchem --artifacts ecfp graphs
!python -m src.train.train_view --encoder attentivefp --tag _smoke --datasets freesolv --epochs 3 --patience 3 --device cuda
!rm -f results/metrics/*_smoke*.csv results/preds/*_smoke*.npy models/*_smoke*.pt
print('smoke test done and cleaned up')

## 7. Train

One tag, eight datasets, six splits. `--artifacts ecfp graphs` keeps materialisation to what
a graph view actually reads.

In [ ]:
!python -m scripts.run_view_multiseed     --tags attentivefp     --variants deepchem seed0 seed1 seed2 seed3 seed4     --artifacts ecfp graphs     --device cuda --resume --restore none

## 8. Check what finished

In [ ]:
import glob
allok = True
for v in ['deepchem','seed0','seed1','seed2','seed3','seed4']:
    m = len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_attentivefp_*.csv'))
    p = len(glob.glob(f'{OUTDIR}/runs/{v}/preds/*_attentivefp_*.npy'))
    done = m == 16 and p == 16
    allok &= done
    print(v, 'metrics', m, 'preds', p, 'ok' if done else 'INCOMPLETE - re-run cell 7')
print()
print('ALL DONE - run cell 9' if allok else 'Not finished yet. Re-run cell 7.')

## 9. Bring the results home

Then, locally, **with `--dry-run` first**:

```
python -m scripts.merge_colab_results ~/Downloads/phase4_results.zip --dry-run
```

`attentivefp` has no committed counterpart, so this merge adds rows rather than colliding
with any -- unlike the Phase 3 regeneration, it can be applied directly.

In [ ]:
import shutil, os
out = shutil.make_archive('/content/phase4_results', 'zip', f'{OUTDIR}/runs')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)